In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# RESUMEN

0. Información importante
1. Normalización
2. Factorización ponderada de matrices (WMF)
3. Evaluación

# 0. Importante

Ejecutar el codigo almenos 1 vez para tener todos los archivos necesarios!

# 1. Normalización

In [2]:
df = pd.read_csv("BBDD_100K/ratings.csv")

indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
matriz_usuario_pelicula=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')

def normalizar_datos(matriz_escasez):
    # Creamos una copia de la matriz para evitar modificar el original
    matriz_escasez_copy = matriz_escasez.copy()
    
    # Inicializamos StandardScaler sin centrado en 0 debido a NaNs
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Aplicamos la normalización solo en las columnas que tienen datos no NaN
    for user_id in matriz_escasez_copy.index:
        # Seleccionamos las calificaciones del usuario (excluyendo NaNs)
        user_ratings = matriz_escasez_copy.loc[user_id].dropna()
        if not user_ratings.empty:
            # Normalizamos las calificaciones de este usuario
            normalized_ratings = scaler.fit_transform(user_ratings.values.reshape(-1, 1)).flatten()
            # Colocamos los valores normalizados en la matriz original, manteniendo NaNs donde no hay calificaciones
            matriz_escasez_copy.loc[user_id, user_ratings.index] = normalized_ratings
    
    # Llenamos los NaNs con 0
    matriz_escasez_copy = matriz_escasez_copy.fillna(0)
    return matriz_escasez_copy

matriz_normalizada = normalizar_datos(matriz_usuario_pelicula)
matriz_normalizada.head(20)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.458937,0.000000,-0.458937,0.000000,0.000000,-0.458937,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.371391,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.596225,1.773677,-0.581226,1.773677,0.596225,0.596225,-0.581226,0.0,-0.581226,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.958138,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.000000,0.442374,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-1.636784,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 2. Factorización ponderada de matrices (WMF)

In [3]:
# Función para inicializar los factores de usuario y película
def inicializar_factores(num_usuarios, num_items, num_factors):
    U = np.random.normal(scale=0.01, size=(num_usuarios, num_factors))
    V = np.random.normal(scale=0.01, size=(num_items, num_factors))
    return U, V

# Función para aplicar WMF
def factorizacion_ponderada_SGD_con_mascara(matriz, num_factors, num_iteraciones, learning_rate, regularizacion, output_csv):
    num_usuarios, num_items = matriz.shape
    U, V = inicializar_factores(num_usuarios, num_items, num_factors)
    # Crear una máscara donde las entradas existentes tienen peso 1, las faltantes 0
    mascara = (matriz != 0).astype(float)
    for iteracion in range(num_iteraciones):
        for i in range(num_usuarios):
            for j in range(num_items):
                # Actualizar solo las entradas observadas
                if mascara[i, j] == 1:
                    error = matriz[i, j] - np.dot(U[i, :], V[j, :])
                    U[i, :] += learning_rate * (error * V[j, :] - regularizacion * U[i, :])
                    V[j, :] += learning_rate * (error * U[i, :] - regularizacion * V[j, :])
        # Calcular el error cuadrático medio solo para las entradas observadas
        mse = np.mean((mascara * (matriz - (U @ V.T))) ** 2)
        print(f"Iteración {iteracion + 1}/{num_iteraciones}, MSE: {mse:.4f}")
    # Generar las predicciones completas
    predicciones_completas = np.dot(U, V.T)
    predicciones_df = pd.DataFrame(predicciones_completas, index=matriz_normalizada.index, columns=matriz_normalizada.columns)
    predicciones_df.to_csv(output_csv, index=True)
    print(f"Predicciones guardadas en {output_csv}")
    return U, V, predicciones_df

Vamos a predecir los valores faltantes

In [4]:
# Parámetros
output_csv = "FPM_100K/100K_usuario_pelicula_datos_simulados.csv"
num_factors = 10          # Número de factores latentes
num_iteraciones = 50      # Número de iteraciones
learning_rate = 0.05      # Tasa de aprendizaje
regularizacion = 0.1      # Parámetro de regularización

# Convertimos la matriz normalizada a numpy array
matriz_numpy = matriz_normalizada.values

# Aplicamos la factorización ponderada
U, V, predicciones_simuladas_df = factorizacion_ponderada_SGD_con_mascara(matriz_numpy, num_factors, num_iteraciones, learning_rate, regularizacion,output_csv)
predicciones_completas = np.dot(U, V.T)
# Crear una máscara para identificar las entradas faltantes
mascara = (matriz_numpy != 0).astype(float)
# Predicciones solo para las entradas faltantes
predicciones_simuladas = (1 - mascara) * predicciones_completas
# Convertimos a DataFrame para visualizar mejor
predicciones_simuladas_df = pd.DataFrame(predicciones_simuladas, index=matriz_normalizada.index, columns=matriz_normalizada.columns)
predicciones_simuladas_df.head()

Iteración 1/50, MSE: 0.0170
Iteración 2/50, MSE: 0.0167
Iteración 3/50, MSE: 0.0151
Iteración 4/50, MSE: 0.0141
Iteración 5/50, MSE: 0.0136
Iteración 6/50, MSE: 0.0133
Iteración 7/50, MSE: 0.0130
Iteración 8/50, MSE: 0.0126
Iteración 9/50, MSE: 0.0122
Iteración 10/50, MSE: 0.0118
Iteración 11/50, MSE: 0.0114
Iteración 12/50, MSE: 0.0110
Iteración 13/50, MSE: 0.0106
Iteración 14/50, MSE: 0.0103
Iteración 15/50, MSE: 0.0099
Iteración 16/50, MSE: 0.0096
Iteración 17/50, MSE: 0.0093
Iteración 18/50, MSE: 0.0091
Iteración 19/50, MSE: 0.0089
Iteración 20/50, MSE: 0.0087
Iteración 21/50, MSE: 0.0085
Iteración 22/50, MSE: 0.0084
Iteración 23/50, MSE: 0.0083
Iteración 24/50, MSE: 0.0082
Iteración 25/50, MSE: 0.0082
Iteración 26/50, MSE: 0.0081
Iteración 27/50, MSE: 0.0080
Iteración 28/50, MSE: 0.0080
Iteración 29/50, MSE: 0.0079
Iteración 30/50, MSE: 0.0079
Iteración 31/50, MSE: 0.0079
Iteración 32/50, MSE: 0.0078
Iteración 33/50, MSE: 0.0078
Iteración 34/50, MSE: 0.0078
Iteración 35/50, MSE: 0

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,-0.172835,-0.000000,-0.998094,-0.047836,0.000000,-0.430496,-0.870330,-0.780924,-0.305053,...,-0.073891,-0.243989,0.104009,0.110602,-0.074172,0.112971,-0.059553,-0.065975,-0.069317,0.113641
2,-0.076987,-0.617620,-0.032124,-0.839848,-0.261153,-0.168433,-0.237493,0.322882,0.513636,-0.085774,...,-0.018940,-0.056117,0.027387,0.009358,-0.019861,0.039584,-0.021823,-0.012862,-0.026374,-0.024340
3,0.126003,-0.228901,-0.344173,0.696669,-0.771452,0.157023,-0.318426,0.331167,0.343120,0.383082,...,-0.007332,-0.015930,0.000441,-0.005136,-0.016035,0.007980,-0.021205,-0.004560,-0.016915,0.060992
4,-0.026858,0.279510,-0.590954,0.001932,0.071799,-0.059182,-0.273205,-0.150147,-0.159651,-0.214333,...,-0.011439,-0.050932,0.032052,0.026126,-0.018868,0.012501,-0.014401,-0.019427,-0.003921,0.068558
5,0.000000,0.257816,0.202826,-1.091377,-0.465499,0.194596,0.021846,-0.145111,-0.917532,-0.081832,...,-0.057932,-0.218976,0.107997,0.094603,-0.051831,0.090664,-0.059465,-0.076610,-0.073476,0.085436


Juntamos ambas matrices

In [4]:
matriz_simulada = pd.read_csv("FPM_100K/100K_usuario_pelicula_datos_simulados.csv", index_col = 0)
# Aseguramos que los índices y columnas coincidan
matriz_simulada.columns = matriz_normalizada.columns
matriz_simulada.index = matriz_normalizada.index

matriz_completa = matriz_normalizada.copy()
matriz_completa = matriz_completa.where(matriz_completa != 0, matriz_simulada)

matriz_completa.head(6)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.458937,-0.172835,-0.458937,-0.998094,-0.047836,-0.458937,-0.430496,-0.870330,-0.780924,-0.305053,...,-0.073891,-0.243989,0.104009,0.110602,-0.074172,0.112971,-0.059553,-0.065975,-0.069317,0.113641
2,-0.076987,-0.617620,-0.032124,-0.839848,-0.261153,-0.168433,-0.237493,0.322882,0.513636,-0.085774,...,-0.018940,-0.056117,0.027387,0.009358,-0.019861,0.039584,-0.021823,-0.012862,-0.026374,-0.024340
3,0.126003,-0.228901,-0.344173,0.696669,-0.771452,0.157023,-0.318426,0.331167,0.343120,0.383082,...,-0.007332,-0.015930,0.000441,-0.005136,-0.016035,0.007980,-0.021205,-0.004560,-0.016915,0.060992
4,-0.026858,0.279510,-0.590954,0.001932,0.071799,-0.059182,-0.273205,-0.150147,-0.159651,-0.214333,...,-0.011439,-0.050932,0.032052,0.026126,-0.018868,0.012501,-0.014401,-0.019427,-0.003921,0.068558
5,0.371391,0.257816,0.202826,-1.091377,-0.465499,0.194596,0.021846,-0.145111,-0.917532,-0.081832,...,-0.057932,-0.218976,0.107997,0.094603,-0.051831,0.090664,-0.059465,-0.076610,-0.073476,0.085436
6,0.568769,0.596225,1.773677,-0.581226,1.773677,0.596225,0.596225,-0.581226,-0.745702,-0.581226,...,-0.001432,0.018503,-0.001654,-0.006744,0.028907,-0.003996,0.034537,-0.001762,0.004435,-0.109089


### Vamos a reescalar los valores, tanto los simulados como los normalizados sobre los valores originales

Primero los normalizados

In [9]:
def reescalar_matriz_normalizada(predicciones_normalizadas, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar de cada usuario ignorando los NaNs originales
    medias_usuarios = matriz_usuario_pelicula.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula.std(axis=1, skipna=True)
    
    # Aplicamos la transformación inversa sólo donde había datos originales
    predicciones_reescaladas = predicciones_normalizadas.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    predicciones_reescaladas = predicciones_reescaladas.where(~matriz_usuario_pelicula.isna(), np.nan)
    predicciones_reescaladas = predicciones_reescaladas.round()
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating)
    
    return predicciones_reescaladas

In [10]:
matriz_normalizada_reescalada = reescalar_matriz_normalizada(matriz_normalizada, matriz_usuario_pelicula)
matriz_normalizada_reescalada.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Luego los simulados

In [11]:
def reescalar_matriz_simulada(matriz_simulada, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar de cada usuario ignorando los NaNs originales
    medias_usuarios = matriz_usuario_pelicula.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula.std(axis=1, skipna=True)
    
    # Aplicamos la transformación inversa a toda la matriz simulada
    predicciones_reescaladas = matriz_simulada.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    
    # Redondeamos y limitamos los valores dentro del rango permitido
    predicciones_reescaladas = predicciones_reescaladas.round()
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating)
    
    return predicciones_reescaladas


In [12]:
def reescalar_solo_simulados(matriz_simulada, matriz_usuario_pelicula_original, min_rating=0.5, max_rating=5.0):
    # Crear una máscara de los valores simulados (donde matriz_usuario_pelicula_original tiene NaN)
    mascara_simulados = matriz_usuario_pelicula_original.isna()
    # Calculamos la media y desviación estándar de cada usuario desde la matriz original
    medias_usuarios = matriz_usuario_pelicula_original.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula_original.std(axis=1, skipna=True)
    # Manejar desviaciones estándar NaN o cero reemplazándolas con 1
    desviaciones_usuarios = desviaciones_usuarios.replace(0, 1).fillna(1)

    valores_simulados = matriz_simulada.copy()
    valores_simulados[~mascara_simulados] = np.nan  # Mantener solo los valores simulados

    valores_simulados = valores_simulados.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    valores_simulados = valores_simulados.clip(lower=min_rating, upper=max_rating)
    valores_simulados = valores_simulados.round(2)

    return valores_simulados

In [14]:
# Reescalar solo los valores simulados
valores_simulados_reescalados = reescalar_solo_simulados(
    matriz_simulada=matriz_simulada,
    matriz_usuario_pelicula_original=matriz_usuario_pelicula,
    min_rating=0.5,
    max_rating=5.0
)

valores_simulados_reescalados_a_csv = valores_simulados_reescalados.fillna(0)
valores_simulados_reescalados_a_csv.to_csv("FPM_100K/100K_usuario_pelicula_datos_simulados_reescalados.csv", index=True)

print("Valores Simulados Reescalados (solo simulados, 2 decimales):")
valores_simulados_reescalados.head(6)

Valores Simulados Reescalados (solo simulados, 2 decimales):


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,4.23,NaN,3.57,4.33,NaN,4.02,3.67,3.74,4.12,...,4.31,4.17,4.45,4.45,4.31,4.46,4.32,4.31,4.31,4.46
2,3.89,3.45,3.92,3.27,3.74,3.81,3.76,4.21,4.36,3.88,...,3.93,3.90,3.97,3.96,3.93,3.98,3.93,3.94,3.93,3.93
3,2.70,1.96,1.72,3.89,0.82,2.76,1.77,3.13,3.15,3.24,...,2.42,2.40,2.44,2.43,2.40,2.45,2.39,2.43,2.40,2.56
4,3.52,3.92,2.78,3.56,3.65,3.48,3.20,3.36,3.35,3.27,...,3.54,3.49,3.60,3.59,3.53,3.57,3.54,3.53,3.55,3.65
5,NaN,3.89,3.84,2.56,3.18,3.83,3.66,3.49,2.73,3.56,...,3.58,3.42,3.74,3.73,3.59,3.73,3.58,3.56,3.56,3.72
6,3.98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.86,NaN,...,3.49,3.51,3.49,3.49,3.52,3.49,3.52,3.49,3.50,3.40


Matriz completa

In [11]:
# Aseguramos que las matrices tienen índices y columnas alineados
valores_simulados_reescalados.columns = matriz_normalizada.columns
valores_simulados_reescalados.index = matriz_normalizada.index
matriz_completa_reescalada = matriz_normalizada_reescalada.copy()
matriz_completa_reescalada = matriz_completa_reescalada.where(~pd.isna(matriz_completa_reescalada), valores_simulados_reescalados)
matriz_completa_reescalada.head(6)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.00,4.23,4.00,3.57,4.33,4.00,4.02,3.67,3.74,4.12,...,4.31,4.17,4.45,4.45,4.31,4.46,4.32,4.31,4.31,4.46
2,3.89,3.45,3.92,3.27,3.74,3.81,3.76,4.21,4.36,3.88,...,3.93,3.90,3.97,3.96,3.93,3.98,3.93,3.94,3.93,3.93
3,2.70,1.96,1.72,3.89,0.82,2.76,1.77,3.13,3.15,3.24,...,2.42,2.40,2.44,2.43,2.40,2.45,2.39,2.43,2.40,2.56
4,3.52,3.92,2.78,3.56,3.65,3.48,3.20,3.36,3.35,3.27,...,3.54,3.49,3.60,3.59,3.53,3.57,3.54,3.53,3.55,3.65
5,4.00,3.89,3.84,2.56,3.18,3.83,3.66,3.49,2.73,3.56,...,3.58,3.42,3.74,3.73,3.59,3.73,3.58,3.56,3.56,3.72
6,3.98,4.00,5.00,3.00,5.00,4.00,4.00,3.00,2.86,3.00,...,3.49,3.51,3.49,3.49,3.52,3.49,3.52,3.49,3.50,3.40


Como en el ejemplo anterior con KNN voy a ver si la media de simulacion de los datos simulados para el usuario 442 que tenía originalmente una media de 1.275 se acercan o no

In [12]:
valores_simulados_usuario = valores_simulados_reescalados.loc[442]
puntuaciones_simuladas = valores_simulados_usuario.dropna()
media_simulada = puntuaciones_simuladas.mean()
print(f"Media de puntuación simulada del usuario 442: {media_simulada:.2f}")

Media de puntuación simulada del usuario 442: 1.25


Los datos son muy similares, ahora tocaria mirar el accuracy prediciendo algunos valores originales aleatorios. En concreto voy a seleccionar un 10% aleatorio sobre la máscara de la matriz original para simular datos, ese 10% serán valores originales que voy a simular para posteriormente comparar, si los resultados son satisfactorios, la factorización ponderada de matrices desarrollada será válida

# 3. Evaluación

Primero seleccionamos un 10% aleatorio de datos originales

In [8]:
def generar_factorizacion(matriz_original, mascara_simulacion, num_factors, num_iteraciones, learning_rate, regularizacion, output_csv="FPM_100K/evaluacion_10%.csv"):
    matriz_modificada = matriz_original.copy()
    matriz_modificada[mascara_simulacion] = 0  # Eliminar temporalmente los valores seleccionados para simulación
    matriz_modificada = matriz_modificada.fillna(0)
    U, V,fpm = factorizacion_ponderada_SGD_con_mascara(matriz_modificada.values, num_factors, num_iteraciones, learning_rate, regularizacion,output_csv)
    predicciones_completas = np.dot(U, V.T)
    predicciones_completas_df = pd.DataFrame(predicciones_completas, index=matriz_original.index, columns=matriz_original.columns)
    
    return predicciones_completas_df

In [9]:
def recortar_por_mascara(predicciones_completas, mascara_simulacion):
    """
    Recorta las predicciones completas utilizando la máscara de simulación.
    Devuelve un DataFrame con NaN en las posiciones fuera de la máscara.
    """
    # Crear un DataFrame con NaN en todas las posiciones
    predicciones_recortadas = pd.DataFrame(np.nan, index=predicciones_completas.index, columns=predicciones_completas.columns)
    # Conservar solo las posiciones seleccionadas por la máscara
    predicciones_recortadas[mascara_simulacion] = predicciones_completas[mascara_simulacion]
    return predicciones_recortadas

In [10]:
def reescalar_simulados_con_mascara(matriz_simulada, matriz_usuario_pelicula_original, mascara_simulacion, min_rating=0.5, max_rating=5.0):
    """
    Reescala únicamente los valores simulados seleccionados por la máscara, utilizando las estadísticas de la matriz original.
    """
    # Calculamos la media y desviación estándar de cada usuario desde la matriz original
    medias_usuarios = matriz_usuario_pelicula_original.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula_original.std(axis=1, skipna=True)
    # Manejar desviaciones estándar NaN o cero reemplazándolas con 1
    desviaciones_usuarios = desviaciones_usuarios.replace(0, 1).fillna(1)
    # Crear una copia para trabajar únicamente con los valores simulados según la máscara
    valores_simulados = matriz_simulada.copy()
    valores_simulados[~mascara_simulacion] = np.nan  # Mantener solo los valores seleccionados por la máscara

    # Reescalado
    valores_simulados = valores_simulados.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    valores_simulados = valores_simulados.clip(lower=min_rating, upper=max_rating)
    valores_simulados = valores_simulados.round(2)
    return valores_simulados

In [11]:
num_factors = 10
num_iteraciones = 50
learning_rate = 0.05
regularizacion = 0.1

mascara_original = ~matriz_usuario_pelicula.isna()
num_datos = mascara_original.sum().sum()
num_datos_a_simular = int(0.1 * num_datos)

indices_aleatorios = np.random.choice(
    mascara_original.stack()[mascara_original.stack()].index,
    size=num_datos_a_simular,
    replace=False
)

mascara_simulacion = pd.DataFrame(False, index=matriz_usuario_pelicula.index, columns=matriz_usuario_pelicula.columns)
for fila, columna in indices_aleatorios:
    mascara_simulacion.loc[fila, columna] = True

Vamos a mirar si los valores pertenecen a la máscara

In [17]:
num_valores_validos = mascara_original.sum().sum()
num_simulados = mascara_simulacion.sum().sum()
porcentaje_simulados = (num_simulados / num_valores_validos) * 100

print(f"Total de valores válidos en la matriz original: {num_valores_validos}")
print(f"Total de valores seleccionados para simulación: {num_simulados}")
print(f"Porcentaje de valores simulados: {porcentaje_simulados:.2f}%")

Total de valores válidos en la matriz original: 100836
Total de valores seleccionados para simulación: 10083
Porcentaje de valores simulados: 10.00%


In [18]:
predicciones_completas = generar_factorizacion(
    matriz_original=matriz_normalizada,
    mascara_simulacion=mascara_simulacion,
    num_factors=num_factors,
    num_iteraciones=num_iteraciones,
    learning_rate=learning_rate,
    regularizacion=regularizacion
)

Iteración 1/50, MSE: 0.0153
Iteración 2/50, MSE: 0.0152
Iteración 3/50, MSE: 0.0140
Iteración 4/50, MSE: 0.0129
Iteración 5/50, MSE: 0.0124
Iteración 6/50, MSE: 0.0121
Iteración 7/50, MSE: 0.0118
Iteración 8/50, MSE: 0.0115
Iteración 9/50, MSE: 0.0112
Iteración 10/50, MSE: 0.0108
Iteración 11/50, MSE: 0.0104
Iteración 12/50, MSE: 0.0100
Iteración 13/50, MSE: 0.0096
Iteración 14/50, MSE: 0.0092
Iteración 15/50, MSE: 0.0089
Iteración 16/50, MSE: 0.0086
Iteración 17/50, MSE: 0.0083
Iteración 18/50, MSE: 0.0080
Iteración 19/50, MSE: 0.0078
Iteración 20/50, MSE: 0.0076
Iteración 21/50, MSE: 0.0075
Iteración 22/50, MSE: 0.0074
Iteración 23/50, MSE: 0.0073
Iteración 24/50, MSE: 0.0072
Iteración 25/50, MSE: 0.0071
Iteración 26/50, MSE: 0.0070
Iteración 27/50, MSE: 0.0070
Iteración 28/50, MSE: 0.0069
Iteración 29/50, MSE: 0.0069
Iteración 30/50, MSE: 0.0068
Iteración 31/50, MSE: 0.0068
Iteración 32/50, MSE: 0.0068
Iteración 33/50, MSE: 0.0067
Iteración 34/50, MSE: 0.0067
Iteración 35/50, MSE: 0

In [12]:
predicciones_10_por_ciento = pd.read_csv("FPM_100K/evaluacion_10%.csv",index_col=0)
predicciones_10_por_ciento.columns = predicciones_10_por_ciento.columns.astype(int)

predicciones_recortadas = recortar_por_mascara(predicciones_10_por_ciento, mascara_simulacion)
predicciones_recortadas.head()

,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
valores_reescalados_con_mascara = reescalar_simulados_con_mascara(
    matriz_simulada=predicciones_recortadas,
    matriz_usuario_pelicula_original=matriz_usuario_pelicula,
    mascara_simulacion=mascara_simulacion
)

print("Valores reescalados con máscara:")
valores_reescalados_con_mascara.head()

Valores reescalados con máscara:


,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
def calcular_mape_rmse_solo_simulados(valores_originales, valores_simulados):
    """
    Calcula el MAPE y el RMSE para los valores simulados en comparación con los valores originales.
    """
    # Máscara de valores simulados válidos (no NaN)
    mascara_simulados = ~valores_simulados.isna()
    # Extraer los valores simulados y sus correspondientes originales
    valores_simulados_filtrados = valores_simulados[mascara_simulados]
    valores_originales_filtrados = valores_originales[mascara_simulados]
    # Calcular el MAPE
    errores_relativos = np.abs((valores_simulados_filtrados - valores_originales_filtrados) / valores_originales_filtrados)
    mape = errores_relativos.mean().mean() * 100  # Promedio de errores relativos en porcentaje
    # Calcular el RMSE
    errores_cuadraticos = (valores_simulados_filtrados - valores_originales_filtrados) ** 2
    rmse = np.sqrt(errores_cuadraticos.mean().mean())  # Promedio de errores cuadráticos
    return mape, rmse

In [17]:
mape, rmse = calcular_mape_rmse_solo_simulados(valores_originales=matriz_usuario_pelicula, valores_simulados=valores_reescalados_con_mascara)
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:.4f}")

MAPE: 18.46%
RMSE: 0.5642


Puesto que estamos escogiendo un 10% de datos aleatorios cada ejecución mostrará datos aleatorios, para tener un resultado mas consistente vamos a ejecutar 10 veces para tener una media de MAPE y RMSE

In [12]:
def evaluar_factorizacion_por_semillas(
    matriz_usuario_pelicula, 
    predicciones_10_por_ciento, 
    num_iteraciones=10, 
    porcentaje_simulacion=0.1
):
    """
    Evalúa la factorización ponderada de matrices generando diferentes máscaras de simulación 
    usando semillas aleatorias y calcula el MAPE y RMSE promedio.
    
    Args:
        matriz_usuario_pelicula: DataFrame original de valoraciones de usuarios.
        predicciones_10_por_ciento: DataFrame con predicciones simuladas.
        num_iteraciones: Número de iteraciones (semillas) para ejecutar el proceso.
        porcentaje_simulacion: Porcentaje de datos a simular.

    Returns:
        resultados_df: DataFrame con los resultados de cada iteración (semilla).
        mape_promedio: MAPE promedio de las iteraciones.
        rmse_promedio: RMSE promedio de las iteraciones.
    """
    resultados = []
    
    for seed in range(num_iteraciones):
        np.random.seed(seed)

        # Generar máscara de simulación
        mascara_original = ~matriz_usuario_pelicula.isna()
        num_datos = mascara_original.sum().sum()
        num_datos_a_simular = int(porcentaje_simulacion * num_datos)
        indices_aleatorios = np.random.choice(
            mascara_original.stack()[mascara_original.stack()].index,
            size=num_datos_a_simular,
            replace=False
        )
        mascara_simulacion = pd.DataFrame(False, index=matriz_usuario_pelicula.index, columns=matriz_usuario_pelicula.columns)
        for fila, columna in indices_aleatorios:
            mascara_simulacion.loc[fila, columna] = True
        # Recortar predicciones por la máscara generada
        predicciones_recortadas = recortar_por_mascara(predicciones_10_por_ciento, mascara_simulacion)
        # Reescalar valores
        valores_reescalados_con_mascara = reescalar_simulados_con_mascara(
            matriz_simulada=predicciones_recortadas,
            matriz_usuario_pelicula_original=matriz_usuario_pelicula,
            mascara_simulacion=mascara_simulacion
        )
        # Calcular MAPE y RMSE
        mape, rmse = calcular_mape_rmse_solo_simulados(
            valores_originales=matriz_usuario_pelicula,
            valores_simulados=valores_reescalados_con_mascara
        )
        resultados.append({"Ejecución": seed + 1, "MAPE (%)": round(mape, 2), "RMSE": round(rmse, 4)})

    # Crear DataFrame con los resultados
    resultados_df = pd.DataFrame(resultados)
    mape_promedio = resultados_df["MAPE (%)"].mean()
    rmse_promedio = resultados_df["RMSE"].mean()
    return resultados_df, mape_promedio, rmse_promedio

In [13]:
predicciones_10_por_ciento = pd.read_csv("FPM_100K/evaluacion_10%.csv", index_col=0)
predicciones_10_por_ciento.columns = predicciones_10_por_ciento.columns.astype(int)

resultados_df, mape_promedio, rmse_promedio = evaluar_factorizacion_por_semillas(
    matriz_usuario_pelicula=matriz_usuario_pelicula,
    predicciones_10_por_ciento=predicciones_10_por_ciento,
    num_iteraciones=10,
    porcentaje_simulacion=0.1
)

In [14]:
print("\nResultados por ejecución:")
print(resultados_df.to_string(index=False))
print(f"\nMAPE promedio: {mape_promedio:.2f}%")
print(f"RMSE promedio: {rmse_promedio:.4f}")


Resultados por ejecución:
 Ejecución  MAPE (%)   RMSE
         1     18.51 0.5704
         2     18.02 0.5604
         3     17.55 0.5536
         4     18.05 0.5577
         5     17.51 0.5591
         6     17.41 0.5471
         7     17.88 0.5589
         8     18.23 0.5584
         9     17.26 0.5517
        10     19.16 0.5775

MAPE promedio: 17.96%
RMSE promedio: 0.5595
